# Example DS Experiment — Housing Price Prediction

This notebook demonstrates the kinds of code the MCP Server can detect:
- Model instantiation (RandomForestRegressor, LinearRegression)
- Preprocessing (StandardScaler, SimpleImputer, PCA)
- Metrics (mean_squared_error, r2_score, mean_absolute_error)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Generate synthetic housing data
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'sqft': np.random.randint(500, 3000, n),
    'bedrooms': np.random.randint(1, 6, n),
    'bathrooms': np.random.randint(1, 4, n),
    'age': np.random.randint(0, 50, n),
    'garage': np.random.randint(0, 3, n),
    'price': None
})
df['price'] = (
    150 * df['sqft'] + 10000 * df['bedrooms'] + 8000 * df['bathrooms']
    - 500 * df['age'] + np.random.normal(0, 20000, n)
)

# Introduce some missing values
df.loc[np.random.choice(n, 50), 'age'] = np.nan
print(df.shape, df.isnull().sum())

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Imputation
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Optional PCA
pca = PCA(n_components=4)
X_train_pca = pca.fit_transform(X_train)
print(f'Explained variance: {pca.explained_variance_ratio_.sum():.3f}')

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Baseline: Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('=== Linear Regression ===')
print(f'MSE:  {mean_squared_error(y_test, y_pred_lr):.0f}')
print(f'MAE:  {mean_absolute_error(y_test, y_pred_lr):.0f}')
print(f'R2:   {r2_score(y_test, y_pred_lr):.4f}')

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest ===')
print(f'MSE:  {mean_squared_error(y_test, y_pred_rf):.0f}')
print(f'R2:   {r2_score(y_test, y_pred_rf):.4f}')

In [ ]:
# Feature importances
import matplotlib.pyplot as plt

feature_names = ['sqft', 'bedrooms', 'bathrooms', 'age', 'garage']
importances = rf.feature_importances_

plt.figure(figsize=(8, 4))
plt.barh(feature_names, importances)
plt.title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()